<h2>eda.py</h2>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/cmu/idl/final_project/idl_project
!ls


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ── Config ────────────────────────────────────────────────────────────────────
data_path = r'/content/drive/MyDrive/cmu/idl/final_project/idl_project/6.+Turbofan+Engine+Degradation+Simulation+Data+Set/6. Turbofan Engine Degradation Simulation Data Set/CMAPSSData'

col_names = ['unit', 'cycle', 'os1', 'os2', 'os3'] + [f's{i}' for i in range(1, 22)]
SUBSETS   = ['FD001', 'FD002', 'FD003', 'FD004']

def load_data(file_name):
    return pd.read_csv(os.path.join(data_path, file_name),
                       sep=r'\s+', header=None, names=col_names)

if __name__ == "__main__":
    # Load all four training sets and tag them with their subset label
    dfs = []
    for subset in SUBSETS:
        df = load_data(f'train_{subset}.txt')
        df['subset'] = subset
        max_cycle = df.groupby('unit')['cycle'].max().reset_index()
        max_cycle.columns = ['unit', 'max_cycle']
        df = df.merge(max_cycle, on='unit', how='left')
        df['RUL'] = df['max_cycle'] - df['cycle']
        df.drop('max_cycle', axis=1, inplace=True)
        dfs.append(df)
        print(f"{subset}: {df['unit'].nunique()} engines, {len(df):,} rows, "
              f"max RUL={df['RUL'].max()}, median life={df.groupby('unit')['cycle'].max().median():.0f} cycles")

    all_train = pd.concat(dfs, ignore_index=True)

    # ── Sensor variance across subsets ────────────────────────────────────────
    sensor_cols = [f's{i}' for i in range(1, 22)]
    print("\nPer-sensor std across ALL subsets:")
    print(all_train[sensor_cols].std().sort_values())

    # ── Identify globally flat sensors ────────────────────────────────────────
    flat = [c for c in sensor_cols if all_train[c].std() < 0.01]
    print(f"\nGlobally flat sensors (std < 0.01): {flat}")

    # ── Plot: RUL distribution per subset ─────────────────────────────────────
    fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)
    for ax, (df, subset) in zip(axes, zip(dfs, SUBSETS)):
        life = df.groupby('unit')['cycle'].max()
        ax.hist(life, bins=20, edgecolor='black', color='steelblue', alpha=0.8)
        ax.set_title(f'{subset}\n(n={df["unit"].nunique()} engines)')
        ax.set_xlabel('Engine life (cycles)')
        ax.set_ylabel('Count')
        ax.grid(linestyle='--', alpha=0.5)
    plt.suptitle('Engine Lifetime Distributions — All 4 Subsets', fontsize=13)
    plt.tight_layout()
    plt.savefig('eda_lifetime_distributions.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Saved eda_lifetime_distributions.png")

    # ── Plot: sensor 2 for unit 1 across subsets ──────────────────────────────
    fig, axes = plt.subplots(1, 4, figsize=(16, 3))
    for ax, (df, subset) in zip(axes, zip(dfs, SUBSETS)):
        u1 = df[df['unit'] == 1]
        ax.plot(u1['cycle'], u1['s2'])
        ax.set_title(f'{subset} — Unit 1, s2')
        ax.set_xlabel('Cycle'); ax.set_ylabel('s2')
        ax.grid(linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig('eda_sensor2_unit1.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Saved eda_sensor2_unit1.png")


<h2>preprocessing.py</h2>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os

# ── Config ────────────────────────────────────────────────────────────────────
data_path = r'/content/drive/MyDrive/cmu/idl/final_project/idl_project/6.+Turbofan+Engine+Degradation+Simulation+Data+Set/6. Turbofan Engine Degradation Simulation Data Set/CMAPSSData'

col_names   = ['unit', 'cycle', 'os1', 'os2', 'os3'] + [f's{i}' for i in range(1, 22)]
SUBSETS     = ['FD001', 'FD002', 'FD003', 'FD004']
CLIP_LIMIT  = 125
VAL_FRAC    = 0.20   # 20% of engines per subset → validation set

# Sensors that are constant in every subset (identified in EDA)
DEAD_SENSORS = ['s1', 's5', 's6', 's10', 's16', 's18', 's19']

def load_raw(subset, split):
    """Load train or test file for a given subset."""
    return pd.read_csv(
        os.path.join(data_path, f'{split}_{subset}.txt'),
        sep=r'\s+', header=None, names=col_names
    )

def add_train_rul(df):
    """Compute RUL = max_cycle - current_cycle for training data."""
    max_c = df.groupby('unit')['cycle'].max().reset_index()
    max_c.columns = ['unit', 'max_cycle']
    df = df.merge(max_c, on='unit', how='left')
    df['RUL'] = df['max_cycle'] - df['cycle']
    return df.drop('max_cycle', axis=1)

def add_test_rul(test_df, subset):
    """Compute RUL for test data using ground-truth file."""
    rul_truth = pd.read_csv(
        os.path.join(data_path, f'RUL_{subset}.txt'),
        sep=r'\s+', header=None, names=['true_rul']
    )
    rul_truth['unit'] = rul_truth.index + 1
    max_c = test_df.groupby('unit')['cycle'].max().reset_index()
    max_c.columns = ['unit', 'max_cycle']
    test_df = test_df.merge(max_c, on='unit', how='left')
    test_df = test_df.merge(rul_truth, on='unit', how='left')
    test_df['RUL'] = test_df['true_rul'] + (test_df['max_cycle'] - test_df['cycle'])
    return test_df.drop(['max_cycle', 'true_rul'], axis=1)

if __name__ == "__main__":
    np.random.seed(42)

    train_parts, val_parts, test_parts = [], [], []

    # We fit ONE global scaler on all four training sets combined,
    # so every subset is normalised into the same numeric range.
    # This is critical: FD002/FD004 have 6 operating conditions whose
    # raw sensor values look very different from FD001/FD003.
    # MinMaxScaler maps everything to [0, 1] regardless of regime,
    # giving the GRU a consistent input space.
    all_raw_train_frames = []

    for subset in SUBSETS:
        raw_train = load_raw(subset, 'train')
        raw_train = add_train_rul(raw_train)
        raw_train['subset'] = subset
        all_raw_train_frames.append(raw_train)

    all_raw_train = pd.concat(all_raw_train_frames, ignore_index=True)

    # Drop dead sensors
    drop_cols = DEAD_SENSORS
    all_raw_train.drop(columns=drop_cols, errors='ignore', inplace=True)

    # Identify feature columns (everything except unit, cycle, subset, RUL)
    feature_cols = [c for c in all_raw_train.columns
                    if c not in ('unit', 'cycle', 'subset', 'RUL')]

    # Fit global scaler on all training data
    scaler = MinMaxScaler()
    scaler.fit(all_raw_train[feature_cols])

    # ── Process each subset ───────────────────────────────────────────────────
    for subset in SUBSETS:
        raw_train = all_raw_train[all_raw_train['subset'] == subset].copy()

        # Engine-level 80/20 train/val split
        units       = raw_train['unit'].unique()
        n_val       = max(1, int(len(units) * VAL_FRAC))
        val_units   = np.random.choice(units, size=n_val, replace=False)
        train_units = [u for u in units if u not in val_units]

        tr = raw_train[raw_train['unit'].isin(train_units)].copy()
        vl = raw_train[raw_train['unit'].isin(val_units)].copy()

        # Clip RUL
        tr['RUL'] = tr['RUL'].clip(upper=CLIP_LIMIT)
        vl['RUL'] = vl['RUL'].clip(upper=CLIP_LIMIT)

        # Scale features
        tr[feature_cols] = scaler.transform(tr[feature_cols])
        vl[feature_cols] = scaler.transform(vl[feature_cols])

        # Tag with subset so we can inspect per-subset metrics later
        tr['subset'] = subset
        vl['subset'] = subset

        train_parts.append(tr)
        val_parts.append(vl)

        # ── Test set ─────────────────────────────────────────────────────────
        raw_test = load_raw(subset, 'test')
        raw_test = add_test_rul(raw_test, subset)
        raw_test.drop(columns=drop_cols, errors='ignore', inplace=True)
        raw_test['RUL'] = raw_test['RUL'].clip(upper=CLIP_LIMIT)
        raw_test[feature_cols] = scaler.transform(raw_test[feature_cols])
        raw_test['subset'] = subset
        test_parts.append(raw_test)

        print(f"{subset}: train engines={len(train_units)}, val engines={len(val_units)}, "
              f"test engines={raw_test['unit'].nunique()}")

    # ── Combine and shuffle train / val ───────────────────────────────────────
    train_df = pd.concat(train_parts, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
    val_df   = pd.concat(val_parts,   ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
    test_df  = pd.concat(test_parts,  ignore_index=True)

    # Save
    train_df.to_csv('train_preprocessed.csv', index=False)
    val_df.to_csv('val_preprocessed.csv',   index=False)
    test_df.to_csv('test_preprocessed.csv',  index=False)

    print(f"\nSaved:")
    print(f"  train_preprocessed.csv : {len(train_df):,} rows from {train_df['subset'].unique().tolist()}")
    print(f"  val_preprocessed.csv   : {len(val_df):,} rows from {val_df['subset'].unique().tolist()}")
    print(f"  test_preprocessed.csv  : {len(test_df):,} rows from {test_df['subset'].unique().tolist()}")
    print(f"  Feature columns ({len(feature_cols)}): {feature_cols}")


<h2>model.py</h2>

In [ ]:
import torch
import torch.nn as nn
import numpy as np

class GRUModel(nn.Module):
    """
    Bidirectional GRU for Remaining Useful Life prediction.

    Why GRU instead of LSTM?
    - GRU has 2 gates (update + reset) vs LSTM's 3 (input + forget + output).
    - ~25% fewer parameters for the same hidden size — less risk of overfitting
      when training on a mixed 4-subset dataset.
    - Single hidden state (no separate cell state): simpler gradient flow,
      typically trains faster to the same accuracy on regression tasks.
    - Bidirectional: the forward pass sees early→late degradation trends;
      the backward pass picks up how recent readings compare to earlier ones.
      Both directions concatenated give richer representations.

    Architecture mirrors lstm2 exactly (hidden=128, layers=2, dropout=0.2,
    bidirectional=True) so results are directly comparable.
    """

    def __init__(self, input_size, hidden_size=128, num_layers=2, output_size=1):
        super(GRUModel, self).__init__()

        # Bidirectional GRU — doubles effective hidden dim to hidden_size*2
        self.gru = nn.GRU(
            input_size, hidden_size, num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.2 if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(0.2)

        # hidden_size*2 because bidirectional concatenates fwd + bwd states
        self.fc = nn.Linear(hidden_size * 2, output_size)

    def forward(self, x):
        # x: (batch, window_size, n_features)
        out, _ = self.gru(x)          # out: (batch, window_size, hidden*2)
        out = out[:, -1, :]           # take last timestep: (batch, hidden*2)
        out = self.dropout(out)
        out = self.fc(out)            # (batch, 1)
        return out


def create_sequences(data, window_size):
    """
    Sliding-window sequencing — identical to lstm2.
    Processes each engine separately so windows never cross engine boundaries.
    Drops engines shorter than window_size (front-padding done at eval time).
    """
    sequences, labels = [], []

    for unit in data['unit'].unique():
        unit_data = data[data['unit'] == unit]
        features  = unit_data.drop(columns=['unit', 'RUL', 'subset'], errors='ignore').values
        target    = unit_data['RUL'].values

        if len(unit_data) < window_size:
            continue   # skip very short engines during training

        for i in range(len(unit_data) - window_size + 1):
            sequences.append(features[i : i + window_size])
            labels.append(target[i + window_size - 1])

    return np.array(sequences, dtype=np.float32), np.array(labels, dtype=np.float32)


if __name__ == "__main__":
    print("GRU model definition loaded. Use train.py to train.")


<h2>train.py</h2>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Config (same as lstm2) ────────────────────────────────────────────────────
WINDOW_SIZE   = 50
BATCH_SIZE    = 64
HIDDEN_SIZE   = 128
NUM_LAYERS    = 2
LEARNING_RATE = 0.001
EPOCHS        = 100
PATIENCE      = 15        # early stopping — absent in lstm2, added here
SUBSETS       = ['FD001', 'FD002', 'FD003', 'FD004']

# ── Device: CUDA (Colab GPU) > MPS (Apple Silicon) > CPU ─────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f"Using device: {DEVICE}")

def train_model():
    # ── 1. Load preprocessed data (all 4 subsets already combined & shuffled) ──
    print("Loading preprocessed training data (all 4 subsets)...")
    train_df = pd.read_csv('train_preprocessed.csv')
    val_df   = pd.read_csv('val_preprocessed.csv')

    print(f"Train rows: {len(train_df):,}  |  Val rows: {len(val_df):,}")
    print("Train subset counts:")
    print(train_df['subset'].value_counts().to_string())
    print("Val subset counts:")
    print(val_df['subset'].value_counts().to_string())

    # ── 2. Build sliding-window sequences ─────────────────────────────────────
    print("\nBuilding sequences...")
    X_train, y_train = create_sequences(train_df, WINDOW_SIZE)
    X_val,   y_val   = create_sequences(val_df,   WINDOW_SIZE)

    print(f"Train windows: {len(X_train):,}  |  Val windows: {len(X_val):,}")
    print(f"Input shape per window: {X_train.shape[1:]}")

    # ── 3. Convert to tensors ─────────────────────────────────────────────────
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
    X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
    y_val_t   = torch.tensor(y_val,   dtype=torch.float32).view(-1, 1)

    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t),
                              batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_val_t, y_val_t),
                              batch_size=BATCH_SIZE, shuffle=False)

    # ── 4. Model, loss, optimiser ─────────────────────────────────────────────
    input_size = X_train.shape[2]
    model      = GRUModel(input_size, HIDDEN_SIZE, NUM_LAYERS).to(DEVICE)
    criterion  = nn.SmoothL1Loss()
    optimizer  = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"\nGRU total parameters: {total_params:,}")

    # ── 5. Training loop with validation + early stopping ────────────────────
    train_history, val_history = [], []
    best_val_loss  = float('inf')
    best_state     = None
    patience_count = 0

    print(f"\nStarting Training ({EPOCHS} epochs, patience={PATIENCE})...\n")

    for epoch in range(EPOCHS):
        # — Train —
        model.train()
        epoch_loss = 0.0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            predictions = model(batch_X)
            loss = criterion(predictions, batch_y)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()

        avg_train_loss = epoch_loss / len(train_loader)
        train_history.append(avg_train_loss)

        # — Validate —
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
                val_loss += criterion(model(batch_X), batch_y).item()
        avg_val_loss = val_loss / len(val_loader)
        val_history.append(avg_val_loss)

        print(f"Epoch [{epoch+1:3d}/{EPOCHS}]  "
              f"Train Loss: {avg_train_loss:.4f}  |  "
              f"Val Loss: {avg_val_loss:.4f}  |  "
              f"patience: {patience_count}/{PATIENCE}")

        # — Early stopping —
        if avg_val_loss < best_val_loss:
            best_val_loss  = avg_val_loss
            best_state     = {k: v.clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"\nEarly stopping triggered at epoch {epoch+1}.")
                break

    # ── 6. Restore best weights and save ─────────────────────────────────────
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), 'gru_model.pth')
    print(f"\nBest val loss: {best_val_loss:.4f}")
    print("Model saved as 'gru_model.pth'")

    # ── 7. Plot train vs val loss ─────────────────────────────────────────────
    best_epoch = len(train_history) - patience_count - 1
    plt.figure(figsize=(10, 5))
    plt.plot(train_history, label='Train Loss', color='steelblue')
    plt.plot(val_history,   label='Val Loss (all 4 subsets)', color='darkorange')
    plt.axvline(best_epoch, color='crimson', linestyle=':', label='Best epoch')
    plt.title('GRU Training — All 4 C-MAPSS Subsets')
    plt.xlabel('Epoch')
    plt.ylabel('SmoothL1 Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig('training_loss.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Loss plot saved as 'training_loss.png'")

if __name__ == "__main__":
    train_model()


<h2>evaluate.py</h2>

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

# ── Config (must match train.py) ──────────────────────────────────────────────
WINDOW_SIZE = 50
HIDDEN_SIZE = 128
NUM_LAYERS  = 2
SUBSETS     = ['FD001', 'FD002', 'FD003', 'FD004']

# ── Device: CUDA (Colab GPU) > MPS (Apple Silicon) > CPU ─────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f"Using device: {DEVICE}")

def evaluate_model():
    # ── 1. Load preprocessed test set (all 4 subsets) ─────────────────────────
    print("Loading preprocessed test data (all 4 subsets)...")
    test_df = pd.read_csv('test_preprocessed.csv')

    # ── 2. Build last-window sequences per engine ─────────────────────────────
    # For evaluation we only need the final WINDOW_SIZE cycles of each engine.
    # Short engines are front-padded so no engine is silently dropped.
    X_test, y_test, subsets_test = [], [], []

    for subset in SUBSETS:
        sub_df = test_df[test_df['subset'] == subset]
        for unit in sub_df['unit'].unique():
            unit_data = sub_df[sub_df['unit'] == unit]
            features  = unit_data.drop(columns=['unit', 'RUL', 'subset'], errors='ignore').values
            label     = unit_data['RUL'].iloc[-1]

            if len(features) >= WINDOW_SIZE:
                window = features[-WINDOW_SIZE:]
            else:
                pad    = np.tile(features[0:1], (WINDOW_SIZE - len(features), 1))
                window = np.concatenate([pad, features], axis=0)

            X_test.append(window)
            y_test.append(label)
            subsets_test.append(subset)

    X_test_t   = torch.tensor(np.array(X_test, dtype=np.float32)).to(DEVICE)
    y_test_arr = np.array(y_test, dtype=np.float32)

    # ── 3. Load model ─────────────────────────────────────────────────────────
    input_size = X_test_t.shape[2]
    model = GRUModel(input_size, HIDDEN_SIZE, NUM_LAYERS).to(DEVICE)
    model.load_state_dict(torch.load('gru_model.pth', map_location=DEVICE, weights_only=False))
    model.eval()

    # ── 4. Predict ────────────────────────────────────────────────────────────
    print("Making predictions...")
    with torch.no_grad():
        predictions = model(X_test_t).cpu().numpy().flatten()

    predictions = np.clip(predictions, 0, None)

    # ── 5. Overall RMSE ───────────────────────────────────────────────────────
    overall_rmse = np.sqrt(mean_squared_error(y_test_arr, predictions))
    print(f"\n{'='*50}")
    print(f"  Overall Test RMSE (all 4 subsets): {overall_rmse:.2f}")
    print(f"{'='*50}")

    # ── 6. Per-subset RMSE + bias ─────────────────────────────────────────────
    print("\nPer-subset breakdown:")
    subsets_arr = np.array(subsets_test)
    for subset in SUBSETS:
        mask = subsets_arr == subset
        if mask.sum() == 0:
            continue
        rmse = np.sqrt(mean_squared_error(y_test_arr[mask], predictions[mask]))
        bias = (predictions[mask] - y_test_arr[mask]).mean()
        print(f"  {subset}: RMSE={rmse:.2f}  Bias={bias:+.2f}  n={mask.sum()}")

    # ── 7. PHM asymmetric score ───────────────────────────────────────────────
    def phm_score(y_true, y_pred):
        d = y_pred - y_true
        return float(np.where(d < 0, np.exp(-d / 13.0) - 1, np.exp(d / 10.0) - 1).sum())

    print("\nPHM Asymmetric Score (lower = better):")
    total_score = 0.0
    for subset in SUBSETS:
        mask = subsets_arr == subset
        if mask.sum() == 0:
            continue
        s = phm_score(y_test_arr[mask], predictions[mask])
        total_score += s
        print(f"  {subset}: {s:.0f}")
    print(f"  Overall:  {total_score:.0f}")

    # ── 8. Scatter: predicted vs actual, coloured by subset ───────────────────
    colors = {'FD001': 'steelblue', 'FD002': 'darkorange',
              'FD003': 'seagreen',  'FD004': 'mediumpurple'}
    plt.figure(figsize=(8, 8))
    for subset in SUBSETS:
        mask = subsets_arr == subset
        plt.scatter(y_test_arr[mask], predictions[mask],
                    label=subset, alpha=0.6, s=25, color=colors[subset])
    lim = max(y_test_arr.max(), predictions.max()) + 5
    plt.plot([0, lim], [0, lim], '--', color='gray', lw=1, label='Perfect')
    plt.xlim(0, lim); plt.ylim(0, lim)
    plt.xlabel('True RUL (cycles)')
    plt.ylabel('Predicted RUL (cycles)')
    plt.title(f'GRU — All 4 Subsets  (Overall RMSE={overall_rmse:.2f})')
    plt.legend(markerscale=1.5)
    plt.grid(linestyle='--', alpha=0.5)
    plt.savefig('evaluation_scatter.png', dpi=120, bbox_inches='tight')
    plt.show()

    # ── 9. Per-subset RMSE bar chart ──────────────────────────────────────────
    subset_rmses = []
    for subset in SUBSETS:
        mask = subsets_arr == subset
        subset_rmses.append(np.sqrt(mean_squared_error(y_test_arr[mask], predictions[mask])))

    plt.figure(figsize=(7, 4))
    bars = plt.bar(SUBSETS, subset_rmses,
                   color=[colors[s] for s in SUBSETS], edgecolor='black', alpha=0.85)
    for bar, val in zip(bars, subset_rmses):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=10)
    plt.axhline(overall_rmse, color='crimson', linestyle='--',
                label=f'Overall RMSE={overall_rmse:.2f}')
    plt.ylabel('Test RMSE (cycles)')
    plt.title('Per-Subset Test RMSE — GRU (All 4 Subsets)')
    plt.legend(); plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig('evaluation_per_subset.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Saved evaluation_scatter.png and evaluation_per_subset.png")

if __name__ == "__main__":
    evaluate_model()
